# sub1quant activation-weighted Pareto scan

End-to-end experiment: collect per-layer activation stats, scan mixed-budget at multiple target BPW values, build the winning checkpoints, and run full WikiText PPL on each. Designed to find the BPW/PPL cliff.

Outputs go to `/content/sub1quant/eval_results/act_weighted_scan/`. Each stage is idempotent and resumable. The script logs to `logs/` so you can poll progress from the colab bridge.

## 0. Mount Drive (recommended)

Mount Drive so the scan/build/ppl artifacts survive a runtime reset or disconnect. The colab L4 free-tier idle timeout is ~5-6h; the full pipeline takes ~60-90 minutes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/sub1quant_runs'
import os
os.makedirs(DRIVE_ROOT, exist_ok=True)
print('Drive root:', DRIVE_ROOT)

## 1. Clone the repo and install deps

In [ ]:
REPO_URL = 'https://github.com/<YOUR_USER>/sub1quant.git'  # TODO: replace with your fork
%cd /content
if not os.path.exists('sub1quant'):
    !git clone $REPO_URL sub1quant
%cd /content/sub1quant
!git pull --rebase --autostash
!pip install -q --upgrade 'transformers>=5.5.0' torch accelerate safetensors huggingface_hub

## 2. Get the model

The base model is `google/gemma-4-E2B`. Cache it under `/content/models/`.

In [ ]:
from huggingface_hub import snapshot_download
import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'  # disable if hf_transfer is not installed
if not os.path.exists('/content/models/gemma-4-E2B/model.safetensors'):
    snapshot_download(
        'google/gemma-4-E2B',
        local_dir='/content/models/gemma-4-E2B',
    )
print('Model ready at /content/models/gemma-4-E2B')

## 3. Make sure WikiText-2 is in place

In [ ]:
import os
WIKITEXT = '/content/sub1quant/data/wiki.test.txt'
if not os.path.exists(WIKITEXT):
    !mkdir -p /content/sub1quant/data
    !wget -q https://wikitext.smerity.com/wikitext-2-v1.zip -O /tmp/wikitext.zip
    !unzip -o -q /tmp/wikitext.zip -d /tmp/wikitext
    !cp /tmp/wikitext/wikitext-2-v1/test-00000-of-00001.parquet $WIKITEXT 2>/dev/null || true
    # Fallback: use HuggingFace dataset
    from datasets import load_dataset
    ds = load_dataset('wikitext', 'wikitext-2-raw-v1', split='test')
    with open(WIKITEXT, 'w', encoding='utf-8') as f:
        for row in ds:
            f.write(row['text'] + '\n')
print('WikiText at', WIKITEXT, 'size', os.path.getsize(WIKITEXT))

## 4. Run the activation-weighted Pareto scan

Targets: 3.75, 3.5, 3.25, 3.0, 2.75, 2.5 BPW. The script will:
  1. Collect per-layer activation stats from a 32k-token forward pass (resumable, writes to Drive)
  2. Run scan_mixed_budget at each target with those weights (resumable via layers.jsonl)
  3. Build the checkpoint for each (resumable via shards/)
  4. Run full WikiText PPL on each
  5. Write `summary.json` and `REPORT.txt`

Each stage can be polled independently. Re-running continues from where it left off.

In [ ]:
%cd /content/sub1quant
!python3 -u experiments/act_weighted_pareto_scan.py \
    --model-dir /content/models/gemma-4-E2B \
    --wikitext /content/sub1quant/data/wiki.test.txt \
    --out-dir /content/sub1quant/eval_results/act_weighted_scan \
    --targets 3.75,3.5,3.25,3.0,2.75,2.5 \
    --group-size 128 \
    --act-tokens 32768 \
    --act-max-length 512 \
    --act-stride 512 \
    --device cuda 2>&1 | tee /content/run.log

## 5. (Optional) Poll progress from the colab bridge

If you're driving this from a Windows machine via the colab bridge, the bridge's exec can poll these:

In [ ]:
!tail -n 80 /content/run.log
!echo '---'
!ls -la /content/sub1quant/eval_results/act_weighted_scan/scans/ 2>/dev/null
!ls -la /content/sub1quant/eval_results/act_weighted_scan/checkpoints/ 2>/dev/null
!ls -la /content/sub1quant/eval_results/act_weighted_scan/ppl/ 2>/dev/null

## 6. Final report

In [ ]:
!cat /content/sub1quant/eval_results/act_weighted_scan/REPORT.txt